# Kansas Soil Water Atlas: page

Writes `index.html` (from `template.html`) and the download files in `files/`. It reads only small tables, so it runs in seconds: rerun it after any text change. The figures come from `atlas.ipynb`.

Chapter titles, lead paragraphs, figure titles and captions are in the `CHAPTERS` list below; the intro, methods and layout are in `template.html`.

In [ ]:
import os, json
ROOT = r'D:\OneDrive - Kansas State University\Research projects\kansas-1km'   # the model output (faw/, soil/, boundaries/) is read from here
SEC  = r'D:\OneDrive - Kansas State University\Research projects\kansas-soil-water-atlas'   # this repository: page, figures, code and data
DATA, IMG, FILES = (os.path.join(SEC, d) for d in ('data', 'img', 'files'))
os.chdir(ROOT)
import numpy as np, pandas as pd, rasterio

def nice_date(d):
    "AP-style date, e.g. Oct. 24, 2022."
    m = ['Jan.', 'Feb.', 'March', 'April', 'May', 'June', 'July', 'Aug.', 'Sept.', 'Oct.', 'Nov.', 'Dec.'][d.month - 1]
    return f'{m} {d.day}, {d.year}'

wk    = pd.read_csv(os.path.join(DATA, 'weeks.csv'), parse_dates=['start', 'end'])
daily = pd.read_csv(os.path.join(DATA, 'statewide_daily.csv'), parse_dates=['date'])
valid = np.load(os.path.join(DATA, 'faw_days_yearly.npz'))['valid']
with rasterio.open('faw/faw_20020101.tif') as src: shape, transform = src.shape, src.transform
lat = transform.f + transform.e * (np.arange(shape[0]) + 0.5)
area = np.cos(np.deg2rad(np.broadcast_to(lat[:, None], shape)))[valid] * (abs(transform.a) * 111.32) * (abs(transform.e) * 111.32)
with rasterio.open('soil/taw.tif') as src: taw = src.read(1)[valid]
CAPACITY = float(np.sum(taw * area) * 1e-6)                     # km3 at field capacity everywhere
MAF = 0.810714                                                 # million acre-feet per km3

years = np.arange(wk.year.min(), wk.year.max() + 1)
CY = np.array([y for y in years if (wk.year == y).sum() == 52])  # complete calendar years
LATEST, LAST_DAY = int(years[-1]), daily.date.iloc[-1]
daily['year'] = daily.date.dt.year
daily['frac'] = (daily.date.dt.dayofyear - 1) / (365 + daily.date.dt.is_leap_year)
annual = daily.groupby('year')[['deficit', 'surplus']].sum()
annual['complete'] = annual.index.isin(CY)
annual['median_faw'] = daily.groupby('year').p50.median()
events = pd.read_csv(os.path.join(FILES, 'drought_events.csv'), parse_dates=['start', 'end', 'peak_week'])   # written by atlas.ipynb
ev = events
print(f'{years[0]}-{LATEST} ({len(CY)} complete years), last day {LAST_DAY:%Y-%m-%d}; capacity {CAPACITY:.1f} km3; {len(ev)} drought events')

## Page text, downloads and `index.html`

In [ ]:
import html as H_
BUILD = pd.Timestamp.today()
V = BUILD.strftime('%Y%m%d')                                    # cache buster: browsers fetch new images after each update
# The page shows WebP copies of the figures (3-5 times smaller than PNG at quality 90); the download links keep the PNGs.
from PIL import Image
SIZE = {}
for f in sorted(os.listdir(IMG)):
    if not f.endswith('.png'): continue
    png, webp = os.path.join(IMG, f), os.path.join(IMG, f[:-4] + '.webp')
    im = Image.open(png); SIZE[f[:-4]] = im.size
    if not os.path.exists(webp) or os.path.getmtime(webp) < os.path.getmtime(png):   # only figures that changed
        im.convert('RGB').save(webp, 'WEBP', quality=90, method=6)
for f in os.listdir(IMG):                                                              # drop copies of removed figures
    if f.endswith('.webp') and f[:-5] not in SIZE: os.remove(os.path.join(IMG, f))
img = lambda name: f'img/{name}.webp?v={V}'
wh  = lambda name: f'width="{SIZE[name][0]}" height="{SIZE[name][1]}"'               # reserves the space before the image arrives
ordinal = lambda n: f'{n}{"th" if 11 <= n % 100 <= 13 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")}'

comp = annual[annual.complete]
yD, yS = int(comp.deficit.idxmax()), int(comp.surplus.idxmax())
i0, i1 = daily.paw_km3.idxmin(), daily.paw_km3.idxmax()
stats = [('dry', f'{comp.deficit.max():.0f}', f'deficit-days in {yD}, the highest annual total since 2002'),
         ('wet', f'{comp.surplus.max():.0f}', f'surplus-days in {yS}, the highest annual total since 2002'),
         ('', f'{daily.paw_km3[i0]:.1f}<small> km³</small>', f'of plant available water stored on {nice_date(daily.date[i0])}, the lowest value of the record'),
         ('', f'{daily.paw_km3[i1]:.1f}<small> km³</small>', f'stored on {nice_date(daily.date[i1])}, the highest value of the record, relative to {CAPACITY:.1f} km³ at field capacity')]
STATS = ''.join(f'<div class="stat {c}"><div class="n">{n}</div><div class="t">{t}</div></div>' for c, n, t in stats)

# ---- fixed text of each chapter: (id, number, title, lead paragraphs, figures [(file, title, subtitle, caption, narrow)]) ----
ppt_lo, ppt_hi = 430, 1200
CHAPTERS = [
 ('glance', 'Statewide conditions at a glance', [
   'Each row of the heat maps represents a year and each cell a week. Kansas soils typically dry from summer through fall and recharge from late fall through spring, so most years show FAW values below 0.5 in the second half of the year. The prolonged dry periods of 2011–2012 and 2022–2023 resulted in the largest annual deficits of the record, whereas the wettest springs were observed in 2007, 2016, and 2019.'],
  [('glance_weekly_median', 'Weekly statewide median FAW', 'Each cell represents one week, from 2002 to the present', f'Weeks are 7-day periods starting on Jan. 1. Blank cells correspond to weeks after {nice_date(LAST_DAY)}.', False),
   ('glance_weekly_faw_days', 'Weekly deficit-days and surplus-days', 'Statewide FAW-days accumulated in each week', 'A week with FAW = 0 across the entire state adds 3.5 deficit-days (i.e., 7 days × 0.5).', False),
   ('glance_annual_ledger', 'Annual deficit-days and surplus-days', 'Statewide totals per calendar year', f'The black line represents the net balance (i.e., surplus-days minus deficit-days). Most years had both deficits and surpluses. For instance, {yS} still had {annual.deficit[yS]:.0f} deficit-days.', False)]),
 ('reservoir', 'Plant available water stored in the soil', [
   f'Multiplying FAW by the plant available water capacity of each soil (i.e., the water held between field capacity and the permanent wilting point) results in the volume of plant available water stored across the state. At field capacity, the top 60 cm of Kansas soils hold about {CAPACITY:.0f} km³ ({CAPACITY * MAF:.0f} million acre-feet) of plant available water. Since 2002, statewide storage ranged from {daily.paw_km3.min():.0f} to {daily.paw_km3.max():.0f} km³.'],
  [('reservoir_timeseries', 'Daily plant available water storage, 2002–present', 'Statewide total for the top 60 cm', 'The horizontal line represents storage at field capacity. The right axis shows storage in million acre-feet.', False),
   ('reservoir_where', 'Spatial distribution of stored water', 'Plant available water in the weeks with the lowest and highest statewide storage of the record', 'Both maps share the same color scale.', False)]),
 ('years', 'Annual cycles since 2002', [
   'Each polar chart represents one year, read clockwise starting on Jan. 1 at the top. The black line is the daily statewide median FAW, and the radial axis ranges from FAW = 0 at the center to FAW = 1 at the outer circle. The shaded areas represent the shortfall below FAW = 0.5 (orange) and the excess above FAW = 0.5 (blue). The numbers below each year are the annual deficit-days and surplus-days.'],
  [('years_dials', 'Annual cycles of statewide FAW', 'Daily statewide median FAW for each year since 2002', f'* {LATEST} through {nice_date(LAST_DAY)}.', False),
   ('years_running_totals', 'Cumulative deficit-days and surplus-days', 'Running totals for each calendar year; the three years with the highest totals and the current year are labeled', 'Gray lines represent the remaining years.', False),
   ('years_rings', 'Weekly statewide median FAW in polar form', 'Same data as the weekly heat map, with 2002 at the center and the most recent year at the outer ring', 'Each ring represents one year and each wedge one week.', True)]),
 ('space', 'Spatial patterns', [
   f'Mean annual rainfall in Kansas ranges from about {ppt_lo:,} mm in the far west to {ppt_hi:,} mm in the southeast, a nearly threefold difference that shapes the spatial pattern of soil water. In a typical year, most deficit-days accumulate in western Kansas and most surplus-days in eastern Kansas. However, drought years extend across the entire state.'],
  [('space_typical_year', 'Deficit-days and surplus-days in a typical year', 'Median annual totals at each 1-km pixel, 2002–2025', 'Maps were smoothed with a Savitzky–Golay filter over about 15 km.', False),
   ('space_west_east', 'West-to-east gradient of FAW over time', 'Mean FAW of each north–south strip of Kansas, from the Colorado border (bottom) to the Missouri border (top)', 'Each column represents one week.', False),
   ('space_area_classes', 'Fraction of Kansas in each FAW class', 'Weekly share of the state area', 'Red classes represent the driest conditions.', False)]),
 ('droughts', 'Drought events', [
   f'We identified each drought as one connected region of dry soil (i.e., FAW < 0.2) in space and time. This approach resulted in {len(ev)} events since 2002 that lasted at least 4 weeks and covered at least 5% of Kansas at their peak. In the chart, bubble size represents the peak area and bubble color the total deficit in statewide FAW-days.'],
  [('droughts_events', 'Drought events since 2002', 'Start date and duration of each event', 'Duration is the number of weeks from the first to the last week in which any part of the event had FAW < 0.2. Thus, an event that contracts during winter and expands again the following summer is counted as a single event.', False)]),
]

def figure_html(name, title, sub, cap, narrow):
    return (f'<figure class="{"narrow" if narrow else ""}"><div class="ft">{title}</div><div class="fs">{sub}</div>'
            f'<img src="{img(name)}" {wh(name)} alt="{H_.escape(title)}" loading="lazy" decoding="async">'
            f'<figcaption><span>{cap}</span><a href="img/{name}.png" download>Download PNG</a></figcaption></figure>')

top = events.nlargest(6, 'volume')
table = ('<figure class="narrow"><div class="ft">Largest drought events</div><div class="fs">Ranked by total deficit</div><table class="events"><tr><th>Start</th><th>End</th><th>Weeks</th><th>Peak share of Kansas</th><th>Deficit (FAW-days)</th></tr>'
         + ''.join(f'<tr><td>{nice_date(r.start)}</td><td>{nice_date(r.end)}</td><td>{r.weeks}</td><td>{r.peak:.0f}%</td><td>{r.volume:.1f}</td></tr>' for r in top.itertuples())
         + '</table></figure>')
# the plant available water flyer (white version, metric shown); PDFs in infographics/ for both units
infographic = (f'<figure class="narrow"><div class="ft">How much plant available water Kansas soils can hold</div>'
               f'<div class="fs">Infographic of the plant available water stored in the top 60 cm at field capacity</div>'
               f'<img src="{img("infographic_paw_metric")}" {wh("infographic_paw_metric")} alt="Infographic: plant available water capacity of Kansas soils" loading="lazy" decoding="async">'
               f'<figcaption><span>Download PDF: <a href="infographics/pawc_educational_metric_white.pdf?v={V}" download>metric</a> · '
               f'<a href="infographics/pawc_educational_imperial_white.pdf?v={V}" download>imperial</a></span>'
               f'<a href="img/infographic_paw_metric.png" download>Download PNG</a></figcaption></figure>')
parts = []
for cid, title, leads, figs in CHAPTERS:
    parts.append(f'<section class="chapter" id="{cid}"><h2>{title}</h2><div class="prose">'
                 + ''.join(f'<p>{p}</p>' for p in leads) + '</div>'
                 + (infographic if cid == 'reservoir' else '') + ''.join(figure_html(*f) for f in figs)
                 + (table if cid == 'droughts' else '') + '</section>')
CHAPTER_HTML = '\n'.join(parts)

# ---- one summary per year ----
rD = comp.deficit.rank(ascending=False).astype(int); rS = comp.surplus.rank(ascending=False).astype(int)
YEARS = {}
for y in CY:
    y = int(y); d = daily[daily.year == y]; D, S = annual.loc[y, 'deficit'], annual.loc[y, 'surplus']
    lo, hi_ = d.paw_km3.idxmin(), d.paw_km3.idxmax(); flo, fhi = d.p50.idxmin(), d.p50.idxmax()
    t1 = (f'{D:.0f} deficit-days and {S:.0f} surplus-days, which ranked {ordinal(rD[y])} in deficit-days and {ordinal(rS[y])} in surplus-days '
          f'among the {len(CY)} complete years since 2002.')
    md_ = lambda t: nice_date(t).rsplit(',', 1)[0]                 # month and day only: the year is in the title
    t2 = (f' Statewide storage ranged from {d.paw_km3[lo]:.1f} km³ ({md_(d.date[lo])}) to {d.paw_km3[hi_]:.1f} km³ ({md_(d.date[hi_])}),'
          f' and the statewide median FAW ranged from {d.p50[flo]:.2f} ({md_(d.date[flo])}) to {d.p50[fhi]:.2f} ({md_(d.date[fhi])}).')
    YEARS[str(y)] = dict(title=str(y), text=t1 + t2, img=img(f'year_{y}'), png=f'img/year_{y}.png', w=SIZE[f'year_{y}'][0], h=SIZE[f'year_{y}'][1])

# ---- downloads ----
ann = annual.reset_index().rename(columns={'deficit': 'deficit_days', 'surplus': 'surplus_days', 'median_faw': 'median_statewide_faw'})
ann.insert(3, 'net_faw_days', ann.surplus_days - ann.deficit_days)
ann.round(3).to_csv(os.path.join(FILES, 'annual_faw_days.csv'), index=False)
sd = daily.drop(columns=['year', 'frac']).rename(columns={'paw_km3': 'stored_paw_km3', 'deficit': 'deficit_days', 'surplus': 'surplus_days', 'area_below_ref': 'share_below_0.5'})
sd = sd.rename(columns={f'c{i}': f'share_faw_{i / 10:.1f}_{(i + 1) / 10:.1f}' for i in range(10)})
sd.to_csv(os.path.join(FILES, 'statewide_daily.csv'), index=False, float_format='%.4f', date_format='%Y-%m-%d')
DOWNLOADS = ''.join(f'<li><a href="files/{f}?v={V}" download>{f}</a>: {t}</li>' for f, t in [
    ('statewide_daily.csv', 'statewide values for every day since 2002: area-weighted mean and percentiles (5th to 95th) of FAW, deficit-days, surplus-days, share of Kansas below 0.5 and in each FAW class, and stored plant available water (km³)'),
    ('annual_faw_days.csv', 'deficit-days, surplus-days and net per calendar year, with the statewide median FAW'),
    ('drought_events.csv', 'every drought event in the chart above: start, end, peak week, duration, peak share of Kansas (%) and deficit (statewide FAW-days)')])
DOWNLOADS += '<li>Every figure: use the Download PNG link under it.</li>'

page = open(os.path.join(SEC, 'code', 'template.html'), encoding='utf-8').read()
for k, v in dict(DESCRIPTION=f'Kansas soil water, 2002–{LATEST}: daily plant available water in the top 60 cm (2 ft) of Kansas soils since 2002, mapped at 1-km resolution.',
                 LATEST=str(LATEST), LAST_DAY=nice_date(LAST_DAY), BUILT=nice_date(BUILD), YEAR_BUILT=str(BUILD.year), STATS=STATS,
                 CHAPTERS=CHAPTER_HTML, DOWNLOADS=DOWNLOADS, YEARS_JSON=json.dumps(YEARS)).items():
    page = page.replace(f'%%{k}%%', v)
assert '%%' not in page, page[page.index('%%') - 60: page.index('%%') + 60]
with open(os.path.join(SEC, 'index.html'), 'w', encoding='utf-8') as fh: fh.write(page)
tot = sum(os.path.getsize(os.path.join(IMG, f)) for f in os.listdir(IMG))
print(f'index.html {len(page) / 1e3:.0f} kB; {len(os.listdir(IMG))} images, {tot / 1e6:.1f} MB; files: {os.listdir(FILES)}')